# 4. ICA as a spatial filter: visual and motor components

**Idea.** ICA splits each participant's EEG into components. Each component is a fixed weighting of the electrodes, i.e. a **spatial filter**: its time course is one clean "virtual channel". So far ICA was used to *remove* artifact components. Here it is used to *keep* the brain components we care about (the game has no sound, so no auditory components):

- a **visual component**: map mostly over occipital/parietal electrodes, with an alpha peak (8–12 Hz);
- a **motor component**: map mostly over central (sensorimotor) electrodes, with a mu (8–12 Hz) and/or beta (13–30 Hz) peak. Mu and visual alpha share the same frequencies; only the map tells them apart.

**Selection criterion:** a peak in the band (above the 1/f background) **and** a map over the area, the usual way ICA mu components are chosen. The notebook only **shortlists** candidates and saves them as PNG; the choice is made by eye, with the supervisor, and written into `SPATIAL_FILTER_CHOICE` in `config.py`. Only then are the envelopes computed.

**Steps:** 1. load clean data + ICA → 2. score every kept component (peak × location) → 3. save the candidates as PNG for the review → 4. envelopes of the chosen components (after the review) → 5. save.

Settings (areas, electrodes, bands) are in `config.py` (`SPATIAL_FILTER`).

In [ ]:
import importlib
import os

import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import config as cfg
%matplotlib inline

## 0. Participant

In [ ]:
# execute_notebooks.py sets RT_SUBJECT
SUBJECT = os.environ.get("RT_SUBJECT", "sub-001")
SESSION = cfg.session_of(SUBJECT)
OUT_DIR = cfg.out_dir(SUBJECT)
REVIEW_DIR = cfg.OUT_ROOT / "spatial_filter_review" / SESSION  # one folder per session for the review
REVIEW_DIR.mkdir(parents=True, exist_ok=True)
print(SUBJECT, f"({SESSION})", "->", REVIEW_DIR)

## 1. Load the clean data and the ICA

In [ ]:
raw = mne.io.read_raw(OUT_DIR / f"{SUBJECT}_clean_raw.fif", preload=True)
ica = mne.preprocessing.read_ica(OUT_DIR / f"{SUBJECT}_ica.fif")
removed = pd.read_csv(OUT_DIR / f"{SUBJECT}_badcomps.csv")["component"].astype(int).tolist()
kept = [c for c in range(ica.n_components_) if c not in removed]

sources = ica.get_sources(raw)
print(f"{len(kept)} kept components; removed as artifacts: {removed}")

## 2. Score every kept component

- **Peak (dB):** EEG power falls with frequency (the "1/f" background). A straight line is fitted to the spectrum (log–log) outside the bands; the peak is the highest point of the band above that line (clearly > 0 dB = a real peak; with a broad band like beta, a few dB can be noise, so look at the spectrum). Only task data (not the breaks) is used.
- **Location:** share of the component's (absolute) map weight on the area's electrodes.
- **Motor:** the peak is the larger of mu and beta (both are kept in the table); `motor_side` says whether the map is mostly left (C3), right (C4) or both.

In [ ]:
psd = sources.compute_psd(picks=[f"ICA{c:03d}" for c in kept], fmin=2, fmax=40, reject_by_annotation=True)
freqs, power = psd.freqs, psd.get_data()

# 1/f background: straight line in log-log, fitted outside the bands
log_f, log_p = np.log10(freqs), 10 * np.log10(power)
fit = np.zeros(len(freqs), bool)
for lo, hi in cfg.SPATIAL_FILTER_BACKGROUND:
    fit |= (freqs >= lo) & (freqs <= hi)
slope, offset = np.polyfit(log_f[fit], log_p[:, fit].T, 1)
above = log_p - (slope[:, None] * log_f + offset[:, None])  # dB above the background

maps = np.abs(ica.get_components())  # (channels, components)
ch = ica.info["ch_names"]


def weight(channels):
    """Summed map weight of each kept component on these channels."""
    return maps[[ch.index(c) for c in channels if c in ch]][:, kept].sum(axis=0)


scores = pd.DataFrame({"component": kept})
for area, settings in cfg.SPATIAL_FILTER.items():
    scores[f"{area}_location"] = weight(settings["region"]) / maps[:, kept].sum(axis=0)
    peaks = []
    for band, (lo, hi) in settings["bands"].items():
        scores[f"{area}_{band}_peak_dB"] = above[:, (freqs >= lo) & (freqs <= hi)].max(axis=1)
        peaks.append(f"{area}_{band}_peak_dB")
    scores[f"{area}_peak_dB"] = scores[peaks].max(axis=1)
    scores[f"{area}_peak_band"] = scores[peaks].idxmax(axis=1).str.split("_").str[1]

left, right = weight(cfg.MOTOR_LEFT), weight(cfg.MOTOR_RIGHT)
share_left = left / (left + right)
scores["motor_side"] = np.select([share_left > 0.6, share_left < 0.4], ["left", "right"], "both")
scores.round(2)

## 3. Save the candidates for the review

Per area, the components with the highest peak among those with at least `SPATIAL_FILTER_MIN_LOCATION` of their map in the region (3 per area). If fewer qualify, the next best are shown and marked *outside region*. Each PNG shows the map, the time course over the data, and the spectrum; the title gives the scores.

All participants of a session go into one folder, `spatial_filter_review/<session>/`, with the score table. No component is chosen here.

In [ ]:
for old in REVIEW_DIR.glob(f"{SUBJECT}_*.png"):  # PNGs of an earlier run of this participant
    old.unlink()

candidates = []
for area in cfg.SPATIAL_FILTER:
    ranked = scores.sort_values(f"{area}_peak_dB", ascending=False)
    inside = ranked[f"{area}_location"] >= cfg.SPATIAL_FILTER_MIN_LOCATION
    top = pd.concat([ranked[inside], ranked[~inside]]).head(cfg.SPATIAL_FILTER_N_CANDIDATES)

    for rank, (_, row) in enumerate(top.iterrows(), start=1):
        comp = int(row["component"])
        title = (f"{SUBJECT} {area} #{rank}: ICA{comp:03d} | peak {row[f'{area}_peak_dB']:.1f} dB "
                 f"({row[f'{area}_peak_band']}) | location {row[f'{area}_location']:.0%}")
        if area == "motor":
            title += f" | {row['motor_side']}"
        if row[f"{area}_location"] < cfg.SPATIAL_FILTER_MIN_LOCATION:
            title += " | OUTSIDE REGION"

        figure = ica.plot_properties(raw, picks=[comp], psd_args=dict(fmax=40), show=False)[0]
        figure.suptitle(title, fontsize=9, y=1.04)
        figure.savefig(REVIEW_DIR / f"{SUBJECT}_{area}_{rank}_ICA{comp:03d}.png", dpi=100, bbox_inches="tight")
        plt.close(figure)
        candidates.append({"area": area, "rank": rank, "component": comp})

candidates = pd.DataFrame(candidates)
table = scores.merge(candidates.pivot(index="component", columns="area", values="rank").add_suffix("_candidate"),
                     on="component", how="left")
table.round(3).to_csv(REVIEW_DIR / f"{SUBJECT}_spatial_filter_scores.csv", index=False)
print(candidates.to_string(index=False))

## 4. Amplitude envelopes of the chosen components (after the review)

Write the choice into `SPATIAL_FILTER_CHOICE` in `config.py` as `{area: (component, band)}`, e.g. `"sub-001": {"visual": (5, "alpha"), "motor": (8, "beta")}`, then run from here. Without a choice this step and the next do nothing.

Each chosen component is band-passed, then its Hilbert envelope is taken on the **continuous** signal (no edge artifacts per trial). Components have an arbitrary sign and scale; the envelope removes the sign, and cRQA will use standardized envelopes.

In [ ]:
importlib.reload(cfg)  # pick up changes made to config.py
CHOSEN = cfg.SPATIAL_FILTER_CHOICE.get(SUBJECT, {})
if not CHOSEN:
    print(f"{SUBJECT}: no choice in SPATIAL_FILTER_CHOICE yet; envelopes skipped")

envelopes = {}
for area, (comp, band) in CHOSEN.items():
    x = sources.copy().pick([f"ICA{comp:03d}"])
    x.filter(*cfg.SPATIAL_FILTER[area]["bands"][band], picks="all")  # components are 'misc' channels: picks needed
    x.apply_hilbert(envelope=True, picks="all")
    envelopes[area] = x

if envelopes:
    fig, ax = plt.subplots(figsize=(10, 3))
    t0 = 60
    for area, x in envelopes.items():
        sf = x.info["sfreq"]
        seg = x.get_data(start=int(t0 * sf), stop=int((t0 + 10) * sf))[0]
        comp, band = CHOSEN[area]
        ax.plot(np.linspace(t0, t0 + 10, seg.size), seg / seg.std(), label=f"{area}: ICA{comp:03d}, {band}")
    ax.set(xlabel="time (s)", ylabel="envelope (z)"); ax.legend()

## 5. Save

One file per area, with all events (for cutting trials later), and the choice for the Methods.

In [ ]:
for area, x in envelopes.items():
    comp, band = CHOSEN[area]
    x.save(OUT_DIR / f"{SUBJECT}_{area}_{band}_envelope_raw.fif", overwrite=True)

if CHOSEN:
    choice = pd.DataFrame([{"area": area, "component": comp, "band": band} for area, (comp, band) in CHOSEN.items()])
    choice = choice.merge(scores, on="component", how="left").round(3)
    choice.to_csv(OUT_DIR / f"{SUBJECT}_spatial_filter.csv", index=False)
    choice

## Questions for Anna

1. Motor envelope: mu, beta, or both?
2. One motor component, or one per side (left and right hand)?
3. Choose components per participant by eye, or match them across participants with CORRMAP (a "visual" and a "motor" template), as for the artifacts?
4. SSD (spatio-spectral decomposition; Nikulin et al. 2011, Dähne et al. 2014) instead of ICA components, or as a check?
5. Compare with the simple alternative (average over ROI electrodes) as a check?